# Baseline, Polling and Efficient Polling

**Dataset.** Set `DATASET` in the constants cell: `cifar10` (the paper's
benchmark), `cifar100`, `mnist`, `fashion_mnist`, or `covertype` — 54 tabular
features and no images at all, which swaps the CNN for an MLP and takes the
claim out of vision entirely. Everything downstream — the loaders, the network,
the class count, the divergence threshold, the results directory and the
checkpoint names — follows from it, so a claim can be checked against five
datasets instead of resting on one. The stored outputs below are the CIFAR-10 run.

Runs every method over **five seeds** (`SEEDS = (42, 43, 44, 45, 46)`) and
reports each result as mean ± sample standard deviation. One seed cannot
separate a real difference between two methods from run-to-run noise — with a
0.06 pp gap between Polling and Efficient Polling, that distinction *is* the
claim.

Each seed redraws both the weight initialization and the train/val split, and
every method is re-seeded before it starts, so within a seed the runs differ
only in their learning-rate logic.

**Methods.** Baseline SGD at a fixed `1e-3`; the comparators — Adam, cosine
annealing, step decay, ReduceLROnPlateau, SPS and Armijo; then Polling and
Efficient Polling. Fixed-LR SGD alone is too easy a win to prove anything.

**Efficient Relative Polling.** A further proposal that drops the fixed grid: the user
picks one learning rate and one multiplier, every poll tries three candidates
around the rate in use, the poll interval backs off without a cap and a blow-up
returns training to the best point seen. A `granularities` flag runs it per
batch, per epoch, or both.

**Ablation.** Two further variants replace Efficient Polling's adaptive trigger
with a fixed interval and with a coin flip, both calibrated to the same poll
rate. Polling less often is easy; the claim is that deciding *when* to poll is
what buys the accuracy, and only the ablation can show that.

The algorithms come from the package in `src/` (`efficient_polling_lr_scheduler`),
imported below — this notebook drives them, it does not reimplement them, so
what runs here is what `pip install` ships. Metrics follow one convention for
every method: a batch's training loss and accuracy are measured where its
gradient was taken, *before* the step.

**Cost.** On an RTX 5070, roughly 10 min per seed for each cheap method
(baseline, Adam, the three schedules, SPS ≈ 4 s/epoch), ~13 min for Armijo, and
~22 min for Polling. Budget ~3 h for the three core methods across five seeds,
~1.2 h for the ablation, ~1.5 h for the two Efficient Relative Polling configurations plus
~30 min for its initial-rate runs, and ~10 h for everything. Every (method, seed) is cached as JSON in
`results/<dataset>/`: a restarted kernel picks up where it left off and re-running
a finished cell costs nothing. Delete a JSON file to force that run again.

**Prototyping a new method.** Subclass `PollingOptimizer` and override `step()`:
you inherit exact snapshot/restore (parameters, module buffers and optimizer
state), the optimizer protocol, and the `StepInfo` telemetry `fit()` aggregates.
Set `optimizer_steps` to what the batch actually cost and the cost model stays
honest for free. Add a branch to `build_optimizer()` and it joins the sweep.

> The stored outputs below are from the original single-seed run and predate
> this restructuring. Re-execute to refresh them.


# Imports


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

from IPython.display import HTML
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation

# The methods themselves live in the package (src/efficient_polling_lr_scheduler),
# which is what `pip install efficient-polling-lr-scheduler` gives you. The
# notebook drives them; it does not reimplement them. Install it in editable
# mode from the repository root so edits to src/ take effect here immediately:
#
#     pip install -e ".[examples]"
from efficient_polling_lr_scheduler import (
    ArmijoSGD,
    EfficientPollingSGD,
    PollingSGD,
    EfficientRelativeEpochPolling,
    EfficientRelativePollingSGD,
    SPSSGD,
    accuracy,
    evaluate,
    fit,
)

# Constants


In [ ]:
SEEDS = (42, 43, 44, 45, 46)

# CIFAR-10 is the paper's benchmark. The others are here so a result cannot
# be an artifact of one dataset: CIFAR-100 keeps the images and multiplies the
# classes by ten, MNIST and Fashion-MNIST trade three channels for one and move
# the difficulty in both directions, and Covertype leaves images behind — 54
# tabular features, seven classes, a dense network instead of the CNN.
DATASET = "cifar10"  # cifar10 | cifar100 | mnist | fashion_mnist | covertype

DATA_DIRS = {
    "cifar10": Path("/home/linkezio/Datasets/cifar-10-python/cifar-10-batches-py"),
    "cifar100": Path("/home/linkezio/Datasets/cifar-100-python"),
    "mnist": Path("/home/linkezio/Datasets/MNIST"),
    "fashion_mnist": Path("/home/linkezio/Datasets/fashion-mnist"),
    "covertype": Path("/home/linkezio/Datasets/covertype"),
}

REPO_DIR = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks")

DATA_DIR = DATA_DIRS[DATASET]
MODELS_DIR = REPO_DIR / "models"
RESULTS_DIR = REPO_DIR / "results" / DATASET

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Configs


## Seeds


In [ ]:
# Seeding happens per run, in `run_one()` below, not once for the whole
# notebook: every method is re-seeded to the same value before it builds its
# model, so within a seed the three methods differ only in their learning-rate
# logic, and across seeds they see genuinely different initializations and
# train/val splits.
#
# A single seed cannot tell a real difference between two methods from run-to-run
# noise, which is why every result here is reported over SEEDS as mean ± stdev.
print("seeds:", SEEDS)

## Device


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# Data


## Datasets

In [ ]:
# The loaders live next to the command-line sweep, in
# `examples/benchmark_datasets.py`, so the notebook and the script cannot
# disagree about what they train on. Adding a dataset is one entry there.
import sys

sys.path.insert(0, str(REPO_DIR / "examples"))

from benchmark_datasets import blowup_loss, build_split, compute_mean_std, spec_for

SPEC = spec_for(DATASET)
IMAGES_DIR = REPO_DIR / "images"


def figure_path(stem: str) -> str:
    """CIFAR-10 keeps the file names the paper cites; every other dataset
    appends its own, so a second sweep never overwrites the paper's figures."""
    return str(IMAGES_DIR / f"{stem}{SPEC.figure_suffix}.png")

shape = "x".join(str(d) for d in SPEC.input_shape)
print(f"{SPEC.name}: {SPEC.num_classes} classes, {shape} "
      f"{'images' if SPEC.is_image else 'features'}")
print(f"blow-up threshold: {blowup_loss(SPEC):.4f} "
      f"(2x the cross-entropy of a uniform guess over {SPEC.num_classes} classes)")

## Normalization Statistics


In [ ]:
train_for_stats = build_split(DATASET, DATA_DIR, train=True)

data_mean, data_std = compute_mean_std(train_for_stats)

print("mean:", [round(v, 4) for v in data_mean.flatten().tolist()])
print("std :", [round(v, 4) for v in data_std.flatten().tolist()])

## Train, Validation, Test Split


In [ ]:
class DataLoaderHyperparameters:
    batch_size: int = 64
    val_fraction: float = 0.1
    num_workers: int = 0 # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).

data_loader_hyperparameters = DataLoaderHyperparameters()

In [ ]:
full_train = build_split(DATASET, DATA_DIR, train=True, mean=data_mean, std=data_std)
test_ds = build_split(DATASET, DATA_DIR, train=False, mean=data_mean, std=data_std)


def build_loaders(seed: int) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Loaders for one seed.

    The train/val split is drawn from the seed, so the seeds vary the partition
    as well as the weight initialization -- a difference that survives only in
    both is a difference in the method, not in one lucky split.
    """
    val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
    train_size = len(full_train) - val_size

    train_ds, val_ds = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seed),
    )

    common = dict(
        batch_size=data_loader_hyperparameters.batch_size,
        num_workers=data_loader_hyperparameters.num_workers,
        pin_memory=(DEVICE == "cuda"),
    )
    return (
        DataLoader(train_ds, shuffle=True, **common),
        DataLoader(val_ds, shuffle=False, **common),
        DataLoader(test_ds, shuffle=False, **common),
    )


_train, _val, _test = build_loaders(SEEDS[0])
len(_train.dataset), len(_val.dataset), len(_test.dataset)

# Model


## Hyperparameters


In [ ]:
class ModelHyperparameters:
    epochs: int = 150
    lr: float = 1e-3
    weight_decay: float = 5e-4


model_hyperparameters = ModelHyperparameters()

## Model Class


In [ ]:
class SimpleCNN(nn.Module):
    """The global pool before the classifier means the input side length
    never enters: 28x28 and 32x32 both arrive at the same 256 features, so
    only the channel count and the class count vary between datasets."""

    def __init__(self, in_channels: int = 3, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SimpleMLP(nn.Module):
    """The dense counterpart, for data with no spatial axes to convolve over.
    Same rule: no batch norm and no dropout, so the optimizer stays the only
    source of adaptation."""

    def __init__(self, in_features: int, num_classes: int, widths=(512, 256, 128)):
        super().__init__()
        layers = []
        previous = in_features
        for width in widths:
            layers += [nn.Linear(previous, width), nn.ReLU()]
            previous = width
        layers.append(nn.Linear(previous, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def build_model() -> nn.Module:
    """The network this dataset's shape calls for. The learning-rate logic is
    the object of study, so the architecture is the plainest one that fits."""
    if SPEC.is_image:
        return SimpleCNN(in_channels=SPEC.in_channels, num_classes=SPEC.num_classes)
    (features,) = SPEC.input_shape
    return SimpleMLP(in_features=features, num_classes=SPEC.num_classes)


print(f"{sum(p.numel() for p in build_model().parameters()):,} parameters")

# Train


## Metric Functions


In [ ]:
loss_fn = nn.CrossEntropyLoss()

## Epoch Functions


## Training Runs


In [ ]:
import json
import statistics
import time


class ComparatorHyperparameters:
    """Settings for the methods we compare against.

    `scheduled_lr` is the *top* of the polling candidate set, not the baseline's
    1e-3: a decay schedule that starts where the baseline sits has nothing to
    decay from, and comparing against a deliberately crippled schedule is how a
    paper gets accused of picking a straw man. Cosine, step and plateau all
    start high and come down, which is what they exist to do.

    `sps_max_lr` and `armijo_lr_max` are that same ceiling, so no adaptive method
    may take a step the others were never allowed to consider.
    """

    scheduled_lr: float = 1e-1
    step_size: int = 50          # StepLR over 150 epochs: 1e-1, then 1e-2, 1e-3, 1e-4
    step_gamma: float = 0.1
    plateau_factor: float = 0.5
    plateau_patience: int = 5
    cosine_eta_min: float = 0.0
    sps_max_lr: float = 1e-1
    armijo_lr_max: float = 1e-1
    armijo_alpha: float = 1e-4
    armijo_beta: float = 0.5
    armijo_max_iters: int = 10


comparator_hyperparameters = ComparatorHyperparameters()


class AblationHyperparameters:
    """Trigger ablation: same selection, same guard, a different poll schedule.

    Our contribution is not "poll less", it is *when* to poll. Two controls make
    that testable: a fixed interval and a coin flip. Both are calibrated to the
    poll rate the backoff variant actually measures, so the three variants poll
    equally often and differ only in the rule that decides.

    `poll_rate` is the ~5% reported in the paper. After the `efficient` sweep,
    check the printed `poll_fraction`: if it moved, set `poll_rate` to it and
    re-run the ablation, otherwise the comparison is confounded by cost.
    """

    poll_rate: float = 0.05
    poll_probability: float = poll_rate          # random trigger
    fixed_interval: int = round(1 / poll_rate) - 1  # blind steps between polls


ablation_hyperparameters = AblationHyperparameters()

METHOD_LABELS = {
    "baseline": "SGD (fixed 1e-3)",
    "adam": "Adam (1e-3)",
    "cosine": "SGD + cosine annealing",
    "step": "SGD + step decay",
    "plateau": "SGD + ReduceLROnPlateau",
    "sps": "SPS (Polyak)",
    "armijo": "Armijo line search",
    "polling": "Polling (base paper)",
    "efficient": "Efficient Polling (ours)",
    "efficient_fixed": "  ablation: fixed interval",
    "efficient_random": "  ablation: random trigger",
    "efficient_relative": "Efficient Relative Polling (ours, per batch)",
    "efficient_relative_epoch": "Efficient Relative Polling (ours, per epoch)",
}


def candidate_grid(lr: float) -> tuple[float, ...]:
    """The paper's fixed grid, two decades either side of ``lr``.

    Bit-identical to `polling_hyperparameters.candidate_lrs` at the default rate;
    the initial-rate robustness sweep pins it around another start.
    """
    if lr == model_hyperparameters.lr:
        return polling_hyperparameters.candidate_lrs
    return tuple(float(lr * 10**d) for d in (-2, -1, 0, 1, 2))


def build_optimizer(
    method: str, model: nn.Module, epochs: int, seed: int, lr: float | None = None
):
    """Returns ``(optimizer, scheduler)``; the scheduler is ``None`` for most.

    ``lr`` overrides the base learning rate, for the initial-rate robustness sweep.
    """
    lr = model_hyperparameters.lr if lr is None else lr
    c = comparator_hyperparameters

    if method == "baseline":
        return torch.optim.SGD(model.parameters(), lr=lr), None
    if method == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr), None
    if method == "cosine":
        optim = torch.optim.SGD(model.parameters(), lr=c.scheduled_lr)
        return optim, torch.optim.lr_scheduler.CosineAnnealingLR(
            optim, T_max=epochs, eta_min=c.cosine_eta_min
        )
    if method == "step":
        optim = torch.optim.SGD(model.parameters(), lr=c.scheduled_lr)
        return optim, torch.optim.lr_scheduler.StepLR(
            optim, step_size=c.step_size, gamma=c.step_gamma
        )
    if method == "plateau":
        optim = torch.optim.SGD(model.parameters(), lr=c.scheduled_lr)
        return optim, torch.optim.lr_scheduler.ReduceLROnPlateau(
            optim, mode="min", factor=c.plateau_factor, patience=c.plateau_patience
        )
    if method == "sps":
        return SPSSGD(model, lr=lr, max_lr=c.sps_max_lr), None
    if method == "armijo":
        return (
            ArmijoSGD(
                model,
                lr=lr,
                lr_max=c.armijo_lr_max,
                alpha=c.armijo_alpha,
                beta=c.armijo_beta,
                max_iters=c.armijo_max_iters,
            ),
            None,
        )
    if method == "polling":
        return PollingSGD(model, lr=lr, candidate_lrs=candidate_grid(lr)), None
    if method == "efficient_relative":
        r = efficient_relative_polling_hyperparameters
        return (
            EfficientRelativePollingSGD(
                model,
                lr=lr,
                multiplier=r.multiplier,
                lr_max=r.lr_max,
                spike_z=r.spike_z,
                rollback_loss=r.rollback_loss,
            ),
            None,
        )
    if method == "efficient_relative_epoch":
        # Per epoch the controller drives a plain SGD; see build_epoch_polling().
        return torch.optim.SGD(model.parameters(), lr=lr), None
    if method.startswith("efficient"):
        e = efficient_polling_hyperparameters
        a = ablation_hyperparameters
        # Everything except the trigger is shared, so the ablation cannot be
        # explained by a different guard or a different candidate set.
        kwargs = dict(
            candidate_lrs=candidate_grid(lr),
            spike_factor=e.spike_factor,
            rollback_loss=e.rollback_loss,
            loss_ema_beta=e.loss_ema_beta,
        )
        if method == "efficient":
            kwargs["max_poll_interval"] = e.max_poll_interval
        elif method == "efficient_fixed":
            kwargs.update(trigger="fixed", max_poll_interval=a.fixed_interval)
        elif method == "efficient_random":
            # Seeded per run, so the coin flips differ across seeds the way
            # everything else does -- and never touch the global RNG, which
            # would change this run's batch order.
            kwargs.update(
                trigger="random", poll_probability=a.poll_probability, poll_seed=seed
            )
        else:
            raise ValueError(f"unknown method: {method}")
        return EfficientPollingSGD(model, lr=lr, **kwargs), None
    raise ValueError(f"unknown method: {method}")


def build_epoch_polling(method: str):
    """The epoch-level controller for ``efficient_relative_epoch``; ``None`` for every other method."""
    if method != "efficient_relative_epoch":
        return None
    r = efficient_relative_polling_hyperparameters
    return EfficientRelativeEpochPolling(
        multiplier=r.multiplier, lr_max=r.lr_max, rollback_loss=r.rollback_loss
    )


def run_key(method: str, lr: float | None) -> str:
    """Name of a run: the method, plus the initial rate when it is not the default."""
    return method if lr is None else f"{method}_lr{lr:g}"


def run_one(
    method: str, seed: int, epochs: int | None = None, lr: float | None = None
) -> dict:
    """Train one method on one seed and return everything worth keeping."""
    epochs = model_hyperparameters.epochs if epochs is None else epochs
    key = run_key(method, lr)
    train_loader, val_loader, test_loader = build_loaders(seed)

    # Re-seeded here, so all methods of a seed start from the same weights and
    # see the same batch order.
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = build_model().to(DEVICE)
    optim, scheduler = build_optimizer(method, model, epochs, seed, lr)
    epoch_polling = build_epoch_polling(method)
    checkpoint = MODELS_DIR / f"{DATASET}_best_{key}_seed{seed}.pt"

    print(f"\n=== {key} | seed {seed} | {epochs} epochs ===")
    started = time.perf_counter()
    history = fit(
        model,
        train_loader,
        val_loader,
        optim,
        loss_fn,
        epochs=epochs,
        device=DEVICE,
        checkpoint_path=checkpoint,
        scheduler=scheduler,
        epoch_polling=epoch_polling,
    )
    elapsed = time.perf_counter() - started

    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, DEVICE)

    batches = epochs * len(train_loader)
    return {
        "method": key,
        "seed": seed,
        "epochs": epochs,
        "lr0": model_hyperparameters.lr if lr is None else lr,
        "best_val": history.best_val_acc,
        "best_epoch": history.best_epoch,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "s_per_epoch": elapsed / epochs,
        "poll_fraction": sum(history.polls) / batches if batches else 0.0,
        "steps": sum(history.optimizer_steps),
        "batches": batches,
        "rollbacks": sum(history.rollbacks),
        "spikes": sum(history.spikes),
        "history": history.as_dict(),
    }


def sweep(
    method: str, seeds=SEEDS, epochs: int | None = None, lr: float | None = None
) -> list[dict]:
    """Run one method over every seed, caching each run to RESULTS_DIR.

    A finished (method, seed) is read back from disk instead of retrained, so a
    restarted kernel -- or a second pass after adding a seed -- costs nothing.
    Delete the JSON file to force a re-run.
    """
    key = run_key(method, lr)
    runs = []
    for seed in seeds:
        path = RESULTS_DIR / f"{key}_seed{seed}.json"
        if path.exists():
            print(f"{key} seed {seed}: cached ({path.name})")
            runs.append(json.loads(path.read_text()))
            continue
        result = run_one(method, seed, epochs, lr)
        path.write_text(json.dumps(result, indent=2))
        runs.append(result)
    return runs


def summarize(values: list[float]) -> tuple[float, float]:
    """Mean and sample standard deviation (zero for a single run)."""
    if len(values) == 1:
        return values[0], 0.0
    return statistics.fmean(values), statistics.stdev(values)


def mean_history(runs: list[dict]) -> dict:
    """Element-wise mean over seeds of every per-epoch curve, for plotting."""
    keys = runs[0]["history"].keys()
    return {
        k: np.mean([r["history"][k] for r in runs], axis=0).tolist()
        for k in keys
        if isinstance(runs[0]["history"][k], list)
    }


def band(runs: list[dict], key: str) -> tuple[np.ndarray, np.ndarray]:
    """Min and max across seeds of one curve."""
    stacked = np.asarray([r["history"][key] for r in runs], dtype=float)
    return stacked.min(axis=0), stacked.max(axis=0)

### Baseline


In [ ]:
runs_baseline = sweep("baseline")

### Polling Method


In [ ]:
class PollingHyperparameters:
    candidate_lrs: tuple[float, ...] = (
        float(model_hyperparameters.lr * 10**-2),
        float(model_hyperparameters.lr * 10**-1),
        float(model_hyperparameters.lr),
        float(model_hyperparameters.lr * 10**1),
        float(model_hyperparameters.lr * 10**2),
    )


polling_hyperparameters = PollingHyperparameters()

In [ ]:
runs_polling = sweep("polling")

### Efficient Polling


In [ ]:
class EfficientPollingHyperparameters:
    candidate_lrs: tuple[float, ...] = polling_hyperparameters.candidate_lrs
    max_poll_interval: int = 64
    spike_factor: float = 3.0  # poll immediately if batch loss > factor * EMA
    rollback_loss: float = blowup_loss(SPEC)  # 2x the loss of a uniform guess
    loss_ema_beta: float = 0.9


In [ ]:
efficient_polling_hyperparameters = EfficientPollingHyperparameters()


In [ ]:
runs_efficient_polling = sweep("efficient")

for r in runs_efficient_polling:
    print(
        f"seed {r['seed']}: polls {sum(r['history']['polls'])}/{r['batches']} "
        f"({r['poll_fraction']:.2%}) | steps {r['steps']:,} | "
        f"rollbacks {r['rollbacks']} | spikes {r['spikes']}"
    )

### Trigger ablation

Polling less often is not the contribution — deciding *when* to poll is. This
ablation keeps the selection and the two-tier guard fixed and swaps only rule 1:

| Variant | Trigger |
|---|---|
| **Backoff** (ours) | the interval doubles while the choice is stable, collapses when it changes |
| **Fixed interval** | poll every `K + 1` batches, come what may |
| **Random** | poll each batch with probability `p`, à la sCGQ |

All three are calibrated to the same poll rate (`poll_rate = 5%` → `K = 19`,
`p = 0.05`), so any difference in accuracy is the trigger and not the budget.
If the backoff variant wins here, the adaptive schedule stops being an argument
and becomes a result; if it ties, the honest paper says the rate is what mattered.

Two extra runs per seed, ~1.2 h. The `efficient` sweep below is already cached,
so it costs nothing to include.

In [ ]:
ABLATIONS = ("efficient", "efficient_fixed", "efficient_random")

runs_ablation = {method: sweep(method) for method in ABLATIONS}

print(f"calibration: poll_rate {ablation_hyperparameters.poll_rate:.1%} "
      f"-> K={ablation_hyperparameters.fixed_interval}, "
      f"p={ablation_hyperparameters.poll_probability}\n")

for method, runs in runs_ablation.items():
    acc_m, acc_s = summarize([r["test_acc"] for r in runs])
    poll_m, poll_s = summarize([r["poll_fraction"] for r in runs])
    steps_m, _ = summarize([float(r["steps"]) for r in runs])
    print(
        f"{METHOD_LABELS[method]:32} test acc {acc_m:.2%} ± {acc_s:.2%} | "
        f"polled {poll_m:.2%} ± {poll_s:.2%} | steps {steps_m:,.0f}"
    )

### Comparators

Beating SGD at a fixed `1e-3` proves very little — it is a learning rate nobody
would ship, and a reviewer will say the 84% is reachable with any ordinary
scheduler. These are the methods that make the comparison honest:

| Method | Why it is here |
|---|---|
| **Adam** | The default anyone would actually reach for. |
| **Cosine annealing** | The standard modern schedule. |
| **Step decay** | The classic schedule the base paper's era would use. |
| **ReduceLROnPlateau** | Reacts to the validation curve instead of the epoch counter — the closest scheduler in spirit to polling. |
| **SPS (Polyak)** | Per-step adaptive LR in closed form: does the polling criterion beat arithmetic? |
| **Armijo** | Per-step line search: the classical way to pick a step size by measuring. |

The last two are the family polling actually belongs to, and are the strongest
test of the contribution. All three schedules start at `1e-1`, the top of the
polling candidate set, since a schedule that starts at the baseline's `1e-3` has
nothing to decay from.

Each comparator costs about as much as the baseline (~4 s/epoch, ~10 min per
seed), except Armijo which pays for its line search (~5 s/epoch). Sweep only the
ones you want — `sweep()` caches, so you can add methods later without redoing
anything.


In [ ]:
COMPARATORS = ("adam", "cosine", "step", "plateau", "sps", "armijo")

runs_comparators = {method: sweep(method) for method in COMPARATORS}

for method, runs in runs_comparators.items():
    acc_m, acc_s = summarize([r["test_acc"] for r in runs])
    print(f"{METHOD_LABELS[method]:26} test acc {acc_m:.2%} ± {acc_s:.2%}")

### Efficient Relative Polling

The user chooses one learning rate and one multiplier `m`; every poll tries
`{X/m, X, X·m}` around the rate in use and the winner becomes the new centre,
so the window follows the rate instead of being pinned to a fixed grid; a poll
that comes back blind widens the window by a multiplier for the next one. The
poll interval has no cap: it doubles when a scheduled poll with signal keeps
the rate (a blind poll leaves it unchanged), and a blow-up
sends training back to the best point seen (weights and rate), zeroes the
interval and halves the ceiling of the next slow start, the way TCP handles a
lost packet. Ties keep the current rate, except on the poll right after a
restart, where they go one notch down.

`granularities` is the flag: `"batch"` polls inside `optimizer.step()`
(`EfficientRelativePollingSGD`); `"epoch"` trains a whole epoch per candidate and keeps
the one with the lowest mean training loss, validation judging the best point
(`EfficientRelativeEpochPolling`, driven by `fit()`). `lr_max` keeps the candidates under the same `1e-1` ceiling every
other method gets, so Table I stays fair; set it to `None` to let the window
roam.


In [ ]:
class EfficientRelativePollingHyperparameters:
    granularities: tuple[str, ...] = ("batch", "epoch")  # the flag: which drivers to run
    multiplier: float = 10.0  # {X/m, X, X*m}: one decade apart, like the fixed grid
    lr_max: float | None = comparator_hyperparameters.scheduled_lr  # 1e-1, Table I parity
    spike_z: float = 3.0  # per batch: deviations above the loss trend that force a poll
    rollback_loss: float = blowup_loss(SPEC)  # blow-up bar, as for Efficient Polling


efficient_relative_polling_hyperparameters = EfficientRelativePollingHyperparameters()

EFFICIENT_RELATIVE_METHODS = {"batch": "efficient_relative", "epoch": "efficient_relative_epoch"}


In [ ]:
runs_efficient_relative = {
    granularity: sweep(EFFICIENT_RELATIVE_METHODS[granularity])
    for granularity in efficient_relative_polling_hyperparameters.granularities
}

for granularity, runs in runs_efficient_relative.items():
    acc_m, acc_s = summarize([r["test_acc"] for r in runs])
    poll_m, _ = summarize([r["poll_fraction"] for r in runs])
    restarts_m, _ = summarize([float(r["rollbacks"]) for r in runs])
    print(
        f"{METHOD_LABELS[EFFICIENT_RELATIVE_METHODS[granularity]]:36} "
        f"test acc {acc_m:.2%} ± {acc_s:.2%} | polled {poll_m:.2%} | "
        f"restarts {restarts_m:.1f}"
    )


### Robustness to the initial learning rate

The claim behind Efficient Relative Polling is that the user only has to pick *a*
learning rate, not the right one. Started two decades below or above the
default, Efficient Polling's fixed grid is pinned to `{1e-7 … 1e-3}` or
`{1e-3 … 1e+1}`; the relative window is supposed to walk back to the same
schedule from either side. One seed per start keeps this under an hour; the
`1e-3` start is the main sweep above. The runs are keyed `<method>_lr<start>`
so they never mix with Table I.


In [ ]:
INITIAL_LRS = (1e-5, 1e-1)  # two decades below and above the default 1e-3
ROBUSTNESS_SEEDS = SEEDS[:1]

runs_initial_lr = {
    (method, lr0): sweep(method, seeds=ROBUSTNESS_SEEDS, lr=lr0)
    for lr0 in INITIAL_LRS
    for method in ("efficient", "efficient_relative")
}

for (method, lr0), runs in runs_initial_lr.items():
    acc_m, _ = summarize([r["test_acc"] for r in runs])
    print(f"{method:10} from lr0 = {lr0:g}: test acc {acc_m:.2%}")


In [ ]:
# --- figure vocabulary: one color per method, fixed and never cycled ----------
# Each method keeps its color in every figure. The palette was checked for
# colorblind separation panel by panel rather than picked by eye, and linestyle
# is a second encoding so the panels survive a grayscale print.
METHOD_COLORS = {
    "baseline": "#0072B2",
    "adam": "#D55E00",
    "cosine": "#009E73",
    "step": "#CC79A7",
    "plateau": "#56B4E9",
    "sps": "#E69F00",
    "armijo": "#7570B3",
    "polling": "#E7298A",
    "efficient": "#1B7837",
    "efficient_fixed": "#4575B4",
    "efficient_random": "#BF812D",
    "efficient_relative": "#B2182B",
    "efficient_relative_epoch": "#01665E",
}

METHOD_STYLES = {
    "baseline": "-",
    "adam": "--",
    "cosine": "-.",
    "step": ":",
    "plateau": (0, (3, 1, 1, 1)),
    "sps": "--",
    "armijo": "-.",
    "polling": ":",
    "efficient": "-",
    "efficient_fixed": "--",
    "efficient_random": "-.",
    "efficient_relative": "-",
    "efficient_relative_epoch": "--",
}

# Thirteen curves on one axes is unreadable, so every figure is small multiples:
# one panel per family, at most five series each. `efficient` repeats in the last
# two panels because it is the reference both of them are read against.
METHOD_GROUPS = (
    ("Fixed rate and schedulers", ("baseline", "adam", "cosine", "step", "plateau")),
    ("Step size measured on the batch", ("sps", "armijo", "polling", "efficient")),
    ("Trigger ablation", ("efficient", "efficient_fixed", "efficient_random")),
    ("Efficient relative polling", ("efficient", "efficient_relative", "efficient_relative_epoch")),
)

# The three methods the paper's narrative is built on, for the single-panel
# version of each figure.
CORE_GROUP = (("", ("baseline", "polling", "efficient")),)


def load_all_runs() -> dict[str, list[dict]]:
    """Every cached run on disk, grouped by method, ordered as in METHOD_LABELS.

    Read from RESULTS_DIR rather than from the `runs_*` variables so the figures
    work whatever subset of the sweep cells you ran, and after a kernel restart,
    without retraining anything.
    """
    found: dict[str, list[dict]] = {}
    for path in sorted(RESULTS_DIR.glob("*_seed*.json")):
        record = json.loads(path.read_text())
        found.setdefault(record["method"], []).append(record)
    return {m: found[m] for m in METHOD_LABELS if m in found}


def resolve_groups(runs_per_method: dict[str, list[dict]], groups) -> list:
    """Drop methods with no runs on disk, then drop panels left empty."""
    resolved = [
        (title, [m for m in methods if runs_per_method.get(m)]) for title, methods in groups
    ]
    return [(title, methods) for title, methods in resolved if methods]


def method_label(method: str) -> str:
    return METHOD_LABELS.get(method, method).strip()


def divergence(runs: list[dict], key: str = "val_loss") -> tuple[int | None, int]:
    """First epoch whose loss stops being finite, and how many seeds got there.

    Cosine, step decay and plateau all start at 1e-1 and several of their runs
    blow up around epoch 28. The mean curve inherits the NaN and simply stops,
    which would read as missing data unless the figure says otherwise, so the
    plots below mark the break and count it in the legend.
    """
    firsts = []
    for run in runs:
        bad = np.flatnonzero(~np.isfinite(np.asarray(run["history"][key], dtype=float)))
        if len(bad):
            firsts.append(int(bad[0]) + 1)
    return (min(firsts) if firsts else None), len(firsts)


def diverged_label(method: str, runs: list[dict]) -> tuple[str, int | None]:
    """Legend text for a method, saying so when some of its seeds diverged."""
    first_nan, count = divergence(runs)
    label = method_label(method)
    if count:
        label += f" ({count}/{len(runs)} diverged)"
    return label, first_nan


# Short names for the figures, where a legend has to fit inside a 2-inch panel.
SHORT_LABELS = {
    "baseline": "SGD 1e-3",
    "adam": "Adam",
    "cosine": "cosine",
    "step": "step decay",
    "plateau": "plateau",
    "sps": "SPS",
    "armijo": "Armijo",
    "polling": "Polling",
    "efficient": "Efficient",
    "efficient_fixed": "abl. fixed",
    "efficient_random": "abl. random",
    "efficient_relative": "Eff. relative",
    "efficient_relative_epoch": "Eff. rel. epoch",
}

# The figures are authored at the size they are printed at, so nothing is
# downscaled into illegibility by \includegraphics.
PAPER_RC = {
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6,
    "figure.titlesize": 9,
}
COLUMN_SIZE = (3.45, 2.35)       # one IEEE column, single panel
def column_stack(n_panels: int) -> tuple[float, float]:
    """One column wide, panels stacked. Figures never span both columns, so the
    two-column flow of the paper is never interrupted."""
    return (3.45, 1.75 * n_panels)


def short_label(method: str, runs: list[dict]) -> tuple[str, int | None]:
    """Compact legend text, saying so when some of the method's seeds diverged."""
    first_nan, count = divergence(runs)
    label = SHORT_LABELS.get(method, method)
    if count:
        label += f" ({count}/{len(runs)} div.)"
    return label, first_nan


## Training Animation


### LR Overlay Animation (all methods, symlog scale)

In [ ]:
def animate_lr_overlay(
    runs_per_method: dict[str, list[dict]],
    groups=METHOD_GROUPS,
    interval_ms: int = 80,
    linthresh: float = 1e-4,
) -> HTML:
    """The LR figure, drawn epoch by epoch, with the same panels and colors."""
    panels = resolve_groups(runs_per_method, groups)
    curves = {
        method: np.asarray(mean_history(runs_per_method[method])["lr"], dtype=float)
        for _, methods in panels
        for method in methods
    }

    n_epochs = len(next(iter(curves.values())))
    epochs = np.arange(1, n_epochs + 1)

    fig, axes = plt.subplots(
        len(panels), 1, figsize=(9, 3.2 * len(panels)), sharex=True, constrained_layout=True
    )
    axes = np.atleast_1d(axes)

    artists = []
    for ax, (title, methods) in zip(axes, panels):
        highest = max(float(np.nanmax(curves[m])) for m in methods)
        for method in methods:
            color, style = METHOD_COLORS[method], METHOD_STYLES[method]
            (line,) = ax.plot([], [], color=color, linestyle=style, linewidth=1.6,
                              label=diverged_label(method, runs_per_method[method])[0])
            artists.append((method, line, ax.scatter([], [], color=color, s=30, zorder=5)))
        ax.set_yscale("symlog", linthresh=linthresh)
        ax.set_ylim(0, highest * 1.3 if highest else 1)
        ax.set_ylabel("Learning rate")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
        if title:
            ax.set_title(title, fontsize=10, loc="left")

    axes[-1].set_xlabel("Epoch")
    axes[-1].set_xlim(0.5, n_epochs + 0.5)
    fig.suptitle("Mean learning rate per epoch, all methods")
    marks = [ax.axvline(1, color="gray", ls="--", alpha=0.5) for ax in axes]

    def update(k: int):
        window = slice(0, k + 1)
        shown = epochs[window]
        for method, line, dot in artists:
            values = curves[method][window]
            line.set_data(shown, values)
            dot.set_offsets(np.c_[shown[-1:], values[-1:]])
        for mark in marks:
            mark.set_xdata([float(epochs[k]), float(epochs[k])])
        return []

    anim = animation.FuncAnimation(
        fig, update, frames=n_epochs, interval=interval_ms, blit=False
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
animate_lr_overlay(load_all_runs())


In [ ]:
def save_lr_comparison(
    runs_per_method: dict[str, list[dict]],
    groups=METHOD_GROUPS,
    linthresh: float = 1e-4,
    figsize: tuple[float, float] | None = None,
    save_path: str | None = None,
) -> None:
    """Mean selected LR per epoch, one panel per family, min-max band over seeds."""
    panels = resolve_groups(runs_per_method, groups)
    if not panels:
        raise ValueError("no runs to plot")
    if figsize is None:
        figsize = column_stack(len(panels)) if len(panels) > 1 else COLUMN_SIZE

    first = runs_per_method[panels[0][1][0]]
    n_epochs = len(first[0]["history"]["lr"])
    epochs = np.arange(1, n_epochs + 1)

    with plt.rc_context(PAPER_RC):
        fig, axes = plt.subplots(
            len(panels), 1, figsize=figsize, sharex=True, constrained_layout=True
        )
        axes = np.atleast_1d(axes)

        highest = 0.0
        for ax, (title, methods) in zip(axes, panels):
            for method in methods:
                runs = runs_per_method[method]
                mean = np.asarray(mean_history(runs)["lr"], dtype=float)
                lo, hi = band(runs, "lr")
                label, first_nan = short_label(method, runs)
                ax.plot(
                    epochs, mean,
                    color=METHOD_COLORS[method], linestyle=METHOD_STYLES[method],
                    linewidth=1.2, label=label,
                )
                ax.fill_between(
                    epochs, lo, hi, color=METHOD_COLORS[method], alpha=0.13, linewidth=0
                )
                if first_nan is not None and first_nan > 1:
                    # The schedule keeps producing a rate long after the run is dead.
                    ax.plot(first_nan - 1, mean[first_nan - 2], marker="x", markersize=5,
                            markeredgewidth=1.4, color=METHOD_COLORS[method], zorder=6)
                highest = max(highest, float(np.nanmax(hi)))
            ax.grid(True, which="both", alpha=0.25)
            # 7.5pt, not the 6pt of the other figures: this legend carries the
            # divergence counts, which have to survive the print at column width.
            ax.legend(loc="lower left", fontsize=7.5, framealpha=0.85, handlelength=1.6)
            if title:
                ax.set_title(title, fontsize=7, loc="left", pad=2)

        for ax in axes:
            ax.set_yscale("symlog", linthresh=linthresh)
            ax.set_ylim(0, highest * 1.6 if highest else 1)
            ax.set_ylabel("LR")
        axes[-1].set_xlabel("Epoch")
        axes[-1].set_xlim(0.5, n_epochs + 0.5)

        fig.savefig(save_path or figure_path("training_comparison_LRs_all"), dpi=300, bbox_inches="tight")
        plt.show()
    print(f"Saved: {save_path}")


all_runs = load_all_runs()

# All thirteen methods, faceted by family, sized for a two-column float.
save_lr_comparison(all_runs)

# The three-method version, sized for one column.
save_lr_comparison(
    all_runs, groups=CORE_GROUP, save_path=figure_path("training_comparison_LRs")
)


### Loss Overlay Animation (all methods, symlog scale)

In [ ]:
def animate_loss_overlay(
    runs_per_method: dict[str, list[dict]],
    groups=METHOD_GROUPS,
    interval_ms: int = 80,
    ylim_percentile: tuple[float, float] = (0, 98),
) -> HTML:
    """The loss figure, drawn epoch by epoch. Solid is validation, faint is training."""
    panels = resolve_groups(runs_per_method, groups)
    curves = {
        method: mean_history(runs_per_method[method])
        for _, methods in panels
        for method in methods
    }

    n_epochs = len(next(iter(curves.values()))["val_loss"])
    epochs = np.arange(1, n_epochs + 1)

    fig, axes = plt.subplots(
        len(panels), 1, figsize=(9, 3.2 * len(panels)), sharex=True, constrained_layout=True
    )
    axes = np.atleast_1d(axes)

    artists = []
    for ax, (title, methods) in zip(axes, panels):
        for method in methods:
            color, style = METHOD_COLORS[method], METHOD_STYLES[method]
            (faint,) = ax.plot([], [], color=color, linestyle=style, linewidth=0.9, alpha=0.35)
            (solid,) = ax.plot([], [], color=color, linestyle=style, linewidth=1.6,
                               label=diverged_label(method, runs_per_method[method])[0])
            artists.append((method, faint, solid, ax.scatter([], [], color=color, s=30, zorder=5)))
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
        if title:
            ax.set_title(title, fontsize=10, loc="left")

    joined = np.concatenate(
        [np.asarray(h[k], dtype=float) for h in curves.values() for k in ("train_loss", "val_loss")]
    )
    finite = joined[np.isfinite(joined)]
    low = float(np.percentile(finite, ylim_percentile[0]))
    high = float(np.percentile(finite, ylim_percentile[1]))
    pad = (high - low) * 0.08
    for ax in axes:
        ax.set_ylim(max(0, low - pad), high + pad)

    axes[-1].set_xlabel("Epoch")
    axes[-1].set_xlim(0.5, n_epochs + 0.5)
    fig.suptitle("Loss per epoch, all methods. Faint curves are the training loss.")
    marks = [ax.axvline(1, color="gray", ls="--", alpha=0.5) for ax in axes]

    def update(k: int):
        window = slice(0, k + 1)
        shown = epochs[window]
        for method, faint, solid, dot in artists:
            train = np.asarray(curves[method]["train_loss"], dtype=float)[window]
            val = np.asarray(curves[method]["val_loss"], dtype=float)[window]
            faint.set_data(shown, train)
            solid.set_data(shown, val)
            dot.set_offsets(np.c_[shown[-1:], val[-1:]])
        for mark in marks:
            mark.set_xdata([float(epochs[k]), float(epochs[k])])
        return []

    anim = animation.FuncAnimation(
        fig, update, frames=n_epochs, interval=interval_ms, blit=False
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
animate_loss_overlay(load_all_runs())


In [ ]:
def save_loss_comparison(
    runs_per_method: dict[str, list[dict]],
    groups=METHOD_GROUPS,
    ylim_percentile: tuple[float, float] = (0, 98),
    figsize: tuple[float, float] | None = None,
    save_path: str | None = None,
) -> None:
    """Validation loss (solid, in the legend) over the training loss (faint).

    Every method's training loss is the batch loss measured where its gradient
    was taken, before the step. The package records it that way for all of them,
    so the faint curves are comparable to each other. Only the validation curves
    are named, because ten legend entries per panel would drown the panel.
    """
    panels = resolve_groups(runs_per_method, groups)
    if not panels:
        raise ValueError("no runs to plot")
    if figsize is None:
        figsize = column_stack(len(panels)) if len(panels) > 1 else COLUMN_SIZE

    first = runs_per_method[panels[0][1][0]]
    n_epochs = len(first[0]["history"]["val_loss"])
    epochs = np.arange(1, n_epochs + 1)

    with plt.rc_context(PAPER_RC):
        fig, axes = plt.subplots(
            len(panels), 1, figsize=figsize, sharex=True, constrained_layout=True
        )
        axes = np.atleast_1d(axes)

        everything = []
        for ax, (title, methods) in zip(axes, panels):
            for method in methods:
                runs = runs_per_method[method]
                means = mean_history(runs)
                color, style = METHOD_COLORS[method], METHOD_STYLES[method]

                train = np.asarray(means["train_loss"], dtype=float)
                ax.plot(epochs, train, color=color, linestyle=style, linewidth=0.7, alpha=0.35)

                val = np.asarray(means["val_loss"], dtype=float)
                lo, hi = band(runs, "val_loss")
                label, first_nan = short_label(method, runs)
                ax.plot(epochs, val, color=color, linestyle=style, linewidth=1.2, label=label)
                ax.fill_between(epochs, lo, hi, color=color, alpha=0.12, linewidth=0)
                if first_nan is not None and first_nan > 1:
                    ax.plot(first_nan - 1, val[first_nan - 2], marker="x", markersize=5,
                            markeredgewidth=1.4, color=color, zorder=6)
                everything.extend([train, val])
            ax.grid(True, alpha=0.25)
            ax.legend(loc="upper right", fontsize=6, framealpha=0.85, handlelength=1.6)
            if title:
                ax.set_title(title, fontsize=7, loc="left", pad=2)

        joined = np.concatenate(everything)
        finite = joined[np.isfinite(joined)]
        if len(finite):
            low = float(np.percentile(finite, ylim_percentile[0]))
            high = float(np.percentile(finite, ylim_percentile[1]))
            pad = (high - low) * 0.08
            for ax in axes:
                ax.set_ylim(max(0, low - pad), high + pad)
        for ax in axes:
            ax.set_ylabel("Loss")
        axes[-1].set_xlabel("Epoch")
        axes[-1].set_xlim(0.5, n_epochs + 0.5)

        fig.savefig(save_path or figure_path("training_comparison_losses_all"), dpi=300, bbox_inches="tight")
        plt.show()
    print(f"Saved: {save_path}")


save_loss_comparison(all_runs)

save_loss_comparison(
    all_runs, groups=CORE_GROUP, save_path=figure_path("training_comparison_losses")
)


### Polls per epoch, per trigger

The ablation's argument in one figure: the same budget, spread flat by
the controls and concentrated at the phase transition by the backoff.

In [ ]:
def save_polls_per_epoch(
    runs_per_method: dict[str, list[dict]],
    methods=("efficient", "efficient_fixed", "efficient_random"),
    save_path: str | None = None,
) -> None:
    """Where each trigger spends its polls, under a budget the three share.

    This is the ablation's whole argument in one axes: the same number of polls
    can be spread flat or concentrated where the selection actually changes.
    """
    methods = [m for m in methods if runs_per_method.get(m)]
    if not methods:
        raise ValueError("no ablation runs to plot")

    n_epochs = len(runs_per_method[methods[0]][0]["history"]["polls"])
    epochs = np.arange(1, n_epochs + 1)
    batches_per_epoch = 704
    floor = batches_per_epoch / (efficient_polling_hyperparameters.max_poll_interval + 1)

    with plt.rc_context(PAPER_RC):
        fig, ax = plt.subplots(figsize=COLUMN_SIZE, constrained_layout=True)

        for method in methods:
            runs = runs_per_method[method]
            mean = np.asarray(mean_history(runs)["polls"], dtype=float)
            lo, hi = band(runs, "polls")
            ax.plot(
                epochs, mean,
                color=METHOD_COLORS[method], linestyle=METHOD_STYLES[method],
                linewidth=1.2, label=SHORT_LABELS[method],
            )
            ax.fill_between(epochs, lo, hi, color=METHOD_COLORS[method], alpha=0.12, linewidth=0)

        ax.axhline(floor, color="0.35", linestyle=":", linewidth=1.0)
        ax.annotate(
            f"floor $\\approx$ {floor:.0f}", xy=(3, floor), fontsize=6, color="0.35",
            va="bottom", ha="left",
        )

        ax.set_yscale("log")
        ax.set_xlim(0.5, n_epochs + 0.5)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Polls per epoch")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(loc="upper right", fontsize=6, framealpha=0.85, handlelength=1.6)

        fig.savefig(save_path or figure_path("polls_per_epoch"), dpi=300, bbox_inches="tight")
        plt.show()
    print(f"Saved: {save_path}")


save_polls_per_epoch(all_runs)


### Initial learning rate robustness

Mean selected learning rate per epoch for Efficient Polling and Relative
Polling, one line per starting rate. A fixed grid can only show the schedule
its start allows; the relative window should recover the same schedule from
every start.


In [ ]:
def load_initial_lr_runs() -> dict[tuple[str, float], list[dict]]:
    """Runs keyed by (method, initial rate): the robustness sweep plus the default start."""
    found: dict[tuple[str, float], list[dict]] = {}
    for path in sorted(RESULTS_DIR.glob("*_seed*.json")):
        record = json.loads(path.read_text())
        base = record["method"].split("_lr")[0]
        if base not in ("efficient", "efficient_relative"):
            continue
        lr0 = float(record.get("lr0", model_hyperparameters.lr))
        found.setdefault((base, lr0), []).append(record)
    return dict(sorted(found.items()))


def save_lr_robustness(
    runs_by_start: dict[tuple[str, float], list[dict]],
    save_path: str | None = None,
) -> None:
    """One panel per method, one line per starting rate, log scale."""
    methods = [m for m in ("efficient", "efficient_relative") if any(k[0] == m for k in runs_by_start)]
    if not methods:
        print("no runs on disk yet")
        return
    with mpl.rc_context(PAPER_RC):
        fig, axes = plt.subplots(
            len(methods), 1, figsize=column_stack(len(methods)), sharex=True,
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)
        for ax, method in zip(axes, methods):
            for (m, lr0), runs in runs_by_start.items():
                if m != method:
                    continue
                curve = np.asarray(mean_history(runs)["lr"], dtype=float)
                epochs = np.arange(1, len(curve) + 1)
                _, first_nan = short_label(method, runs)
                ax.plot(epochs, curve, lw=1.2, label=f"start {lr0:g}")
                if first_nan is not None:
                    ax.plot(first_nan, curve[first_nan - 1], "x", ms=5, color="k")
            ax.set_yscale("log")
            ax.set_title(method_label(method))
            ax.set_ylabel("mean LR")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend(loc="lower left")
        axes[-1].set_xlabel("Epoch")
        fig.savefig(save_path or figure_path("initial_lr_robustness"), dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {save_path}")


save_lr_robustness(load_initial_lr_runs())


# Test


In [ ]:
def results_table(runs_per_method: dict[str, list[dict]]) -> None:
    """The paper's table: every column is mean ± sample stdev over the seeds.

    Test metrics come from each run's own best-validation checkpoint, so the
    test set is never used to select anything.
    """
    print("| Method | Best Val | Test Acc | Test Loss | Polled | Steps | s/Epoch | Seeds |")
    print("|---|---|---|---|---|---|---|---|")

    for method, runs in runs_per_method.items():
        if not runs:
            continue
        val_m, val_s = summarize([r["best_val"] for r in runs])
        acc_m, acc_s = summarize([r["test_acc"] for r in runs])
        loss_m, loss_s = summarize([r["test_loss"] for r in runs])
        sec_m, sec_s = summarize([r["s_per_epoch"] for r in runs])
        poll_m, _ = summarize([r["poll_fraction"] for r in runs])
        steps_m, _ = summarize([float(r["steps"]) for r in runs])
        polled = f"{poll_m:.2%}" if poll_m else "n/a"

        print(
            f"| {METHOD_LABELS.get(method, method)} "
            f"| {val_m:.2%} ± {val_s:.2%} "
            f"| {acc_m:.2%} ± {acc_s:.2%} "
            f"| {loss_m:.4f} ± {loss_s:.4f} "
            f"| {polled} "
            f"| {steps_m:,.0f} "
            f"| {sec_m:.2f} ± {sec_s:.2f} "
            f"| {len(runs)} |"
        )


results_table(load_all_runs())
